# Feature Engineering, Encoding & Scaling

## Executive Overview
In this notebook, we transform our cleaned dataset into a numeric, standardized feature matrix suitable for unsupervised machine learning algorithms (specifically **K-Means Clustering**).

### Primary Methodological Decision: Preventing Data Leakage
`Weight`, `Height`, and calculated `BMI` are direct mathematical determinants of the target classes (`NObeyesdad`). Including them in the feature space $X$ during unsupervised clustering would cause structural target leakage—forcing the algorithm to cluster solely on physical mass rather than behavioral profiles.

**Strategy:**
1. **Feature Space ($X$):** Composed strictly of lifestyle, dietary, habits, and genetic indicators.
2. **Validation Targets ($y$):** `BMI` and `NObeyesdad` are isolated to validate and profile the resulting clusters in Notebook 06.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import warnings

warnings.filterwarnings('ignore')

# Load cleaned dataset
df = pd.read_csv('../data/interim/ObesityDataSet_cleaned.csv')

# Calculate BMI solely for post-clustering profiling vector
df['BMI'] = df['Weight'] / (df['Height'] ** 2)

print(f"Dataset successfully loaded. Shape: {df.shape}")

Dataset successfully loaded. Shape: (2111, 18)


## 1. Feature Encoding Strategy

To convert textual and categorical metrics into numerical inputs without introducing artificial ordering biases, we apply three distinct encoding strategies:

* **Binary Encoding ($0 / 1$):** Applied to two-category nominals (`Gender`, `family_history_with_overweight`, `FAVC`, `SMOKE`, `SCC`).
* **Ordinal Frequency Mapping:** Applied to ordered frequency indicators (`CAEC`, `CALC`) to preserve hierarchy ($0$ = `no` to $3$ = `Always`).
* **One-Hot Encoding (Dummy Variables):** Applied to unordered nominal transportation choices (`MTRANS`).

In [2]:
df_encoded = df.copy()

# 1. Binary Mapping
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'yes': 1, 'no': 0})

df_encoded['Gender'] = df_encoded['Gender'].map({'Female': 0, 'Male': 1})

# 2. Ordinal Frequency Mapping
freq_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df_encoded['CAEC'] = df_encoded['CAEC'].map(freq_map)
df_encoded['CALC'] = df_encoded['CALC'].map(freq_map)

# 3. One-Hot Encoding for Unordered Nominals
df_encoded = pd.get_dummies(df_encoded, columns=['MTRANS'], prefix='MTRANS', drop_first=True, dtype=int)

# 4. Target Label Encoding (Reserved for Validation Only)
target_mapping = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}
df_encoded['NObeyesdad_encoded'] = df_encoded['NObeyesdad'].map(target_mapping)

print("Categorical encoding successfully completed.")
print(f"Processed columns count: {df_encoded.shape[1]}")

Categorical encoding successfully completed.
Processed columns count: 22


## 2. Distance-Based Scaling for Clustering

K-Means is a distance-based algorithm relying on Euclidean geometry ($\sqrt{\sum (x_i - y_i)^2}$). Variables measured on larger scales (e.g., `Age` ranging from $14$–$61$) would disproportionately dictate cluster centroids compared to bounded features (e.g., `FAF` ranging from $0$–$3$).

We apply **`StandardScaler`** ($Z$-score normalization) to continuous behavioral metrics:
$$x' = \frac{x - \mu}{\sigma}$$
This centers features at mean $\mu = 0$ with standard deviation $\sigma = 1$, preserving biological outlier distributions (validated during Tukey analysis) while equalizing feature weighting.

In [3]:
# 1. Isolate Pure Behavioral and Genetic Features (Exclude Physical Metric Space)
X_cols = [
    'Age', 'Gender', 'family_history_with_overweight', 
    'FAVC', 'FCVC', 'NCP', 'CAEC', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'SMOKE'
] + [col for col in df_encoded.columns if col.startswith('MTRANS_')]

X = df_encoded[X_cols]

# 2. Isolate Physical Metrics and Targets for Cluster Profiling
y_class = df_encoded['NObeyesdad_encoded']
y_bmi = df_encoded['BMI']

# 3. Apply StandardScaler to Continuous Variables
continuous_cols = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])

# 4. Export Fitted Scaler Object
joblib.dump(scaler, '../models/standard_scaler_clustering.pkl')

print("Feature scaling successfully applied. Sample of standardized feature matrix X:")
X_scaled.head()

Feature scaling successfully applied. Sample of standardized feature matrix X:


,Age,Gender,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,CH2O,SCC,FAF,TUE,CALC,SMOKE,MTRANS_Bike,MTRANS_Motorbike,MTRANS_Public_Transportation,MTRANS_Walking
0,-0.521741,0,1,0,-0.725454,0.522983,1,-0.021330,0,-1.124415,0.497717,0,0,0,0,1,0
1,-0.521741,0,1,0,0.987559,0.522983,1,1.431202,1,2.226606,-0.986295,1,1,0,0,1,0
2,-0.207057,1,1,0,-0.725454,0.522983,1,-0.021330,0,1.109599,0.497717,2,0,0,0,1,0
3,0.422312,1,0,0,0.987559,0.522983,1,-0.021330,0,1.109599,-0.986295,2,0,0,0,0,1
4,-0.364399,1,0,0,-0.725454,-2.209731,1,-0.021330,0,-1.124415,-0.986295,1,0,0,0,1,0


In [4]:
# Save clustering matrix and target validation vectors
X_scaled.to_csv('../data/processed/X_scaled_clustering.csv', index=False)
y_class.to_csv('../data/processed/y_classification.csv', index=False)
y_bmi.to_csv('../data/processed/y_regression_bmi.csv', index=False)

print("All processed artifacts exported successfully to '../data/processed/':")
print(" - X_scaled_clustering.csv (Clustering Input)")
print(" - y_classification.csv (Post-Clustering Label Profiling)")
print(" - y_regression_bmi.csv (Post-Clustering Metric Profiling)")

All processed artifacts exported successfully to '../data/processed/':
 - X_scaled_clustering.csv (Clustering Input)
 - y_classification.csv (Post-Clustering Label Profiling)
 - y_regression_bmi.csv (Post-Clustering Metric Profiling)


---
## Summary & Workflow Transition

1. **Target Leakage Prevention:** Removed `Height`, `Weight`, and `BMI` from feature matrix $X$, ensuring clusters represent purely behavioral/lifestyle profiles.
2. **Encoding Completed:** Transformed ordinal, nominal, and binary attributes into machine-readable numeric formats.
3. **Standardization:** Normalized continuous attributes using `StandardScaler` to prevent scale dominance during distance metrics calculation.

### Next Step: Notebook 06 - Unsupervised Lifestyle Segmentation
We proceed to **`06_unsupervised_clustering.ipynb`** to:
* Evaluate optimal cluster count ($K$) using **Elbow Method** (Inertia) and **Silhouette Analysis**.
* Fit K-Means clustering model on behavioral feature space.
* Map discovered clusters back to `BMI` and `NObeyesdad` to analyze the biological validity of lifestyle segments.